# HDL Genome-Wide Genetic Correlation
**Analysis:** High-Definition Likelihood (HDL) estimation of genetic correlation between congenital septal defects and autism spectrum disorder (ASD).

**Runtime:** R (Google Colab)

---
## 1. Environment Setup

In [ ]:
# Utility wrapper: captures shell output and prints it to the R console
shell_exec <- function(command, ...) {
  output <- system(command, intern = TRUE, ...)
  cat(paste(output, collapse = "\n"), "\n")
  invisible(output)
}

In [ ]:
system("git clone https://github.com/zhenin/HDL.git")

library(devtools)
install_github("zhenin/HDL/HDL")

library(HDL)
library(data.table)
library(dplyr)

---
## 2. Download LD Reference Panels
Using the UKB imputed HapMap2 SVD-based LD reference panel.

In [ ]:
LD_REF_DIR  <- "/content/ld_ref"
LD_REF_FILE <- file.path(LD_REF_DIR, "UKB_imputed_hapmap2_SVD_eigen99_extraction.tar.gz")
LD_REF_URL  <- "https://www.dropbox.com/s/4vuktycxz1an6sp/UKB_imputed_hapmap2_SVD_eigen99_extraction.tar.gz?dl=0"
LD_REF_PATH <- "/content/UKB_imputed_hapmap2_SVD_eigen99_extraction"

dir.create(LD_REF_DIR, showWarnings = FALSE)

In [ ]:
shell_exec(sprintf(
  "wget -c -t 1 '%s' --no-check-certificate -O '%s'",
  LD_REF_URL, LD_REF_FILE
))

In [ ]:
# Verify download integrity
shell_exec(sprintf("md5sum '%s'", LD_REF_FILE))

In [ ]:
shell_exec(sprintf("tar -xzvf '%s' -C /content", LD_REF_FILE))

---
## 3. GWAS Summary Statistics Preparation

### 3.1 Congenital Septal Defects (CHD) — FinnGen Meta-analysis

**Source:** `METAANALYSIS1_Septal.TBL`  
**Total N:** 451,549 (controls) + 2,184 (cases) + 458,440 (replication) = 912,173

In [ ]:
CHD_N_TOTAL <- 451549 + 2184 + 458440
BIM_PATH    <- "/content/g1000_eur/g1000_eur.bim"
SUMMARIES_DIR <- "/content/summaries"

dir.create(SUMMARIES_DIR, showWarnings = FALSE)

chd_gwas <- fread("/content/METAANALYSIS1_Septal.TBL")

In [ ]:
# Standardise column names to HDL conventions and compute Z-score
chd_gwas <- chd_gwas[
  , .(SNP = MarkerName,
      A1  = toupper(Allele1),
      A2  = toupper(Allele2),
      N   = CHD_N_TOTAL,
      B   = Effect,
      SE  = StdErr,
      P   = `P-value`)
]

chd_gwas[, Z := B / SE]

In [ ]:
# Annotate with chromosomal position from 1000G EUR reference panel
bim_ref <- fread(BIM_PATH,
                 col.names = c("CHR", "SNP", "CM", "BP", "A1_ref", "A2_ref"))

chd_gwas <- merge(
  chd_gwas,
  bim_ref[, .(SNP, BP, CHR)],
  by  = "SNP",
  all.x = TRUE
)

saveRDS(chd_gwas, file = "/content/chd_finngen.rds")

rm(bim_ref, chd_gwas)
invisible(gc())

In [ ]:
# Run HDL data wrangling script for CHD GWAS
shell_exec(sprintf(
  "Rscript /content/HDL/HDL.data.wrangling.R \
  gwas.file=%s \
  LD.path=%s \
  SNP=SNP A1=A1 A2=A2 N=N b=B se=SE \
  output.file=%s \
  log.file=%s",
  "/content/chd_finngen.rds",
  LD_REF_PATH,
  file.path(SUMMARIES_DIR, "chd_finngen"),
  file.path(SUMMARIES_DIR, "log_chd")
))

### 3.2 Autism Spectrum Disorder (ASD) — iPSYCH/PGC

**Source:** Grove et al. (2019) iPSYCH-PGC ASD GWAS, November 2017 release  
**Total N:** 18,382 (cases) + 27,969 (controls) = 46,351  
**Note:** Effect allele coding is inverted to match HDL conventions.

In [ ]:
ASD_N_CASES    <- 18382
ASD_N_CONTROLS <- 27969
ASD_N_TOTAL    <- ASD_N_CASES + ASD_N_CONTROLS

shell_exec(
  'wget --user-agent="Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 Chrome/120.0" \
  -O /content/asdPGC.zip \
  "https://figshare.com/ndownloader/articles/14671989/versions/1"'
)
shell_exec("unzip -o /content/asdPGC.zip -d /content/")

In [ ]:
asd_gwas <- fread("/content/iPSYCH-PGC_ASD_Nov2017.gz")

asd_gwas[, N := ASD_N_TOTAL]
asd_gwas[, B := log(OR)]   # convert odds ratio to log-OR effect size
asd_gwas[, Z := B / SE]

# Invert allele coding: A2 (reference) becomes A1 (effect) to match HDL input format
asd_gwas[, c("A1", "A2") := .(A2, A1)]

asd_gwas <- asd_gwas[, .(SNP, CHR, A1, A2, N, Z, B, P, SE, BP)]

saveRDS(asd_gwas, file = "/content/pgc_asd.rds")

rm(asd_gwas)
invisible(gc())

In [ ]:
# Run HDL data wrangling script for ASD GWAS
shell_exec(sprintf(
  "Rscript /content/HDL/HDL.data.wrangling.R \
  gwas.file=%s \
  LD.path=%s \
  SNP=SNP A1=A1 A2=A2 N=N b=B se=SE \
  output.file=%s \
  log.file=%s",
  "/content/pgc_asd.rds",
  LD_REF_PATH,
  file.path(SUMMARIES_DIR, "asd"),
  file.path(SUMMARIES_DIR, "log_asd")
))

---
## 4. HDL Genetic Correlation Estimation

Estimating genome-wide genetic correlation between congenital septal defects (CHD) and autism spectrum disorder (ASD) using the High-Definition Likelihood method (Ning et al., 2020).

In [ ]:
HDL_RESULTS_DIR <- "/content/hdl_results"
dir.create(HDL_RESULTS_DIR, showWarnings = FALSE)

shell_exec(sprintf(
  "Rscript /content/HDL/HDL.run.R \
  gwas1.df=%s \
  gwas2.df=%s \
  LD.path=%s \
  output.file=%s",
  file.path(SUMMARIES_DIR, "chd_finngen.hdl.rds"),
  file.path(SUMMARIES_DIR, "asd.hdl.rds"),
  LD_REF_PATH,
  file.path(HDL_RESULTS_DIR, "chd_asd_genetic_correlation.txt")
))

---
## 5. Results

In [ ]:
results_path <- file.path(HDL_RESULTS_DIR, "chd_asd_genetic_correlation.txt")

if (file.exists(results_path)) {
  cat(readLines(results_path), sep = "\n")
} else {
  message("Results file not found. Check HDL run logs for errors.")
}